# 🚀 Live-Demo: Qdrant Vektordatenbank
---
## Multi-modale Suche & Effizientes Retrieval

In dieser Demo schauen wir uns an, wie eine Vektordatenbank (Qdrant) in der Praxis funktioniert. Wir gehen über die reine Textsuche hinaus und betrachten:

1.  **Vektorisierung:** Wie aus unstrukturierten Daten (Text/Bild) mathematische Vektoren werden.
2.  **Multimodalität:** Speicherung von unterschiedlichen Datentypen in einem gemeinsamen Vektorraum.
3.  **Suchalgorithmen:** Der Performance-Vergleich zwischen exakter Suche (kNN) und approximativer Suche (ANN).
4.  **Filtering:** Wie man Metadaten nutzt, um die semantische Suche einzuschränken.

## 🛠️ 1. Setup & Embedding-Modell

Um Daten „vergleichbar“ zu machen, müssen wir sie in einen Vektorraum einbetten. Wir nutzen hier ein **CLIP-Modell** (`clip-ViT-B-32`).

> **Warum CLIP?** CLIP ist multimodal. Es kann sowohl Bilder als auch Texte in denselben Vektorraum abbilden. Das bedeutet: Ein Bild von einem Hund und das Wort „Hund“ liegen im Vektorraum nah beieinander.

In [ ]:
!pip install qdrant-client sentence-transformers requests pillow

In [ ]:
import time
import requests
from io import BytesIO
from PIL import Image
from IPython.display import display, HTML
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, PayloadSchemaType, SearchParams)

# Client Initialisierung
qdrant_api_key = userdata.get('Qdrant_API_KEY')
client = QdrantClient(
    url="https://473679d4-0dc8-4c97-ab4c-0d9f4c0dab80.europe-west6-0.gcp.cloud.qdrant.io",
    api_key=qdrant_api_key,
)

# Laden des multimodalen Modells
model = SentenceTransformer('clip-ViT-B-32')
collection_name = "Demo_collection"

## 📥 2. Datenspeicherung (Upsert)

In Qdrant speichern wir sogenannte **Points**. Ein Point besteht aus:
*   **ID:** Eindeutiger Identifikator.
*   **Vektor:** Die numerische Repräsentation (das „Verständnis“ des Modells).
*   **Payload:** Zusätzliche Metadaten (JSON), wie Kategorien, URLs oder Originaltexte.



Wir fügen nun live ein Text-Dokument und ein Bild-Embedding hinzu.

In [ ]:
# Neue Daten für die Live-Demo im Notebook
demo_queries_text = [
    {"type": "text", "topic": "Knochen", "content": "Ein klassisches Risotto erfordert ständiges Rühren und die schrittweise Zugabe von heißem Fond."},
    {"type": "text", "topic": "Weltraum", "content": "Die Entdeckung von Exoplaneten in der habitablen Zone befeuert die Suche nach außerirdischem Leben."},
    {"type": "text", "topic": "Sport", "content": "Beim Triathlon müssen die Athleten die Disziplinen Schwimmen, Radfahren und Laufen direkt nacheinander absolvieren."}
]

demo_queries_image = [
    {"topic": "Tiere", "url": "https://images.unsplash.com/photo-1555169062-013468b47731?w=400", "desc": "Ein bunter Papagei im tropischen Regenwald"},
    {"topic": "Architektur", "url": "https://images.unsplash.com/photo-1518780664697-55e3ad937233?w=400", "desc": "Ein modernes Einfamilienhaus in skandinavischem Design"},
    {"topic": "Weltraum", "url": "https://images.unsplash.com/photo-1614728894747-a83421e2b9c9?w=400", "desc": "Die Oberfläche des Mars mit rötlichem Staub und Felsen"}
]

In [ ]:
# Aktuelle Anzahl abrufen
collection_info = client.get_collection(collection_name=collection_name)
current_count = collection_info.points_count

# 1. Text-Dokument einzeln upserten
doc = demo_queries_text[0]
embedding_text = model.encode(doc["content"]).tolist()
point_text = PointStruct(
    id=current_count + 1,
    vector=embedding_text,
    payload=doc
)
client.upsert(collection_name=collection_name, points=[point_text])

# 2. Bild-Metadaten einzeln upserten
img_data = demo_queries_image[0]

# WICHTIG: Das Bild von der URL laden
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(img_data["url"], headers=headers)
img_obj = Image.open(BytesIO(response.content)).convert("RGB")
embedding_img = model.encode(img_obj).tolist()
point_img = PointStruct(
    id=current_count + 2,
    vector=embedding_img,
    payload={
        "type": "image",
        "topic": img_data["topic"],
        "content": img_data["desc"],
        "url": img_data["url"]
    }
)
client.upsert(collection_name=collection_name, points=[point_img])

new_total = client.get_collection(collection_name=collection_name).points_count
print(f"Erfolgreich hinzugefügt: ID {current_count+1} (Text) und ID {current_count+2} (echtes Bild-Embedding).")
print(f"Neue Gesamtzahl: {new_total}")

## 🔍 3. ANN vs. kNN: Geschwindigkeit vs. Präzision

In der Vorlesung haben wir gelernt:
*   **k-Nearest Neighbors (kNN):** Vergleicht die Suchanfrage mit *jedem* einzelnen Vektor. Exakt, aber bei Millionen Daten zu langsam.
*   **Approximate Nearest Neighbors (ANN):** Nutzt Indizes (wie HNSW - Hierarchical Navigable Small World), um nur in der „richtigen Nachbarschaft“ zu suchen. Viel schneller, aber mit minimalem Präzisionsverlust.

Wir messen nun den Zeitunterschied direkt in der Collection.

In [ ]:
# 1. Input festlegen
new_query_doc = demo_queries_text[1]["content"]
query_vector = model.encode(new_query_doc).tolist()

print(f"🔍 EINGABE-QUERY:")
print(f"'{new_query_doc}'")
print("-" * 50)

# --- ANN Suche (Approximative) ---
start_ann = time.time()
res_ann = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=2
).points
time_ann = time.time() - start_ann

# --- kNN Suche (Exakte Suche) ---
start_knn = time.time()
res_knn = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=2,
    search_params=SearchParams(exact=True) # Erzwingt linearen Scan
).points
time_knn = time.time() - start_knn

# --- AUSGABE IN TABELLENFORM ---
def format_result(points):
    if not points: return "Keine Treffer"
    output = ""
    for p in points:
        t = p.payload.get('type', 'N/A').upper()
        c = p.payload.get('content', 'N/A')[:60]
        output += f"<b>[{t}]</b> Score: {p.score:.4f}<br>'{c}...'<br><br>"
    return output

html_table = f"""
<table style="width:100%; border: 1px solid white; border-collapse: collapse;">
  <tr style="background-color: #333; color: white;">
    <th style="padding: 10px; border: 1px solid white;">Algorithmus</th>
    <th style="padding: 10px; border: 1px solid white;">Dauer (Sekunden)</th>
    <th style="padding: 10px; border: 1px solid white;">Top Ergebnisse</th>
  </tr>
  <tr>
    <td style="padding: 10px; border: 1px solid white;"><b>ANN</b> (HNSW Index)</td>
    <td style="padding: 10px; border: 1px solid white;">{time_ann:.5f} s</td>
    <td style="padding: 10px; border: 1px solid white;">{format_result(res_ann)}</td>
  </tr>
  <tr>
    <td style="padding: 10px; border: 1px solid white;"><b>kNN</b> (Exakt)</td>
    <td style="padding: 10px; border: 1px solid white;">{time_knn:.5f} s</td>
    <td style="padding: 10px; border: 1px solid white;">{format_result(res_knn)}</td>
  </tr>
</table>
"""

display(HTML(html_table))

### Ab wann wird ANN schneller?

In der Regel liegt der "Break-even-point" (der Punkt, an dem ANN gewinnt) bei:

* **Vektor-Dimension 512:** Ab ca. 10.000 bis 50.000 Dokumenten.

Erst dann ist die Zeit, die man durch das "Überspringen" von Daten spart, größer als die Zeit, die man für die Verwaltung des Graphen verliert.

## 🔄 4. Update: Daten aktualisieren (PUT & POST-Logik)

In der Welt der Vektordatenbanken gibt es eine Besonderheit im Vergleich zu klassischen relationalen Datenbanken:

* **Upsert (PUT-Logik):** Wenn wir einen `Point` mit einer bereits existierenden ID senden, wird der gesamte Punkt (Vektor + Payload) überschrieben. Existiert die ID noch nicht, wird sie neu angelegt.
* **Set Payload (PATCH-Logik):** Oft möchte man nur die Metadaten (Payload) ändern, ohne den teuren Vektor (das Embedding) neu zu berechnen. Dies spart Rechenzeit und API-Kosten.

**Demo:** Wir überschreiben ID 1 komplett und ändern bei ID 68 nur das Thema.

In [ ]:
client.upsert(
    collection_name=collection_name,
    points=[
        PointStruct(
            id=1,
            vector=model.encode(demo_queries_text[1]["content"]).tolist(),
            payload=demo_queries_text[1]
        )
    ]
)

In [ ]:
client.set_payload(
    collection_name=collection_name,
    payload={
        "topic": "Kochen"
    },
    points=[68]
)

## 🗑️ 5. Delete: Daten entfernen

Das Löschen von Daten ist für die Wartung und den Datenschutz (DSGVO) essenziell. In Qdrant können wir Punkte auf zwei Arten entfernen:
1.  **ID-basiert:** Gezieltes Löschen einzelner Dokumente.
2.  **Filter-basiert:** Löschen aller Dokumente, die ein bestimmtes Kriterium erfüllen (z. B. `topic: "Test"`).

**Demo:** Wir entfernen das Dokument mit der ID 1 aus unserem Index.

In [ ]:
# DELETE: Entferne das Dokument mit ID 2
client.delete(
    collection_name=collection_name,
    points_selector=[1]
)

## 🛡️ 6. Filtered Search (Pre-Filtering)

Ein großer Vorteil von Vektordatenbanken wie Qdrant ist die Fähigkeit, **Vektorsuche mit harten Filtern zu kombinieren**.

Stell dir vor, du suchst nach „Blauwalen“, willst aber nur Ergebnisse aus der Kategorie „Wissenschaft“ sehen.

**Pre-Filtering** sorgt dafür, dass die Vektorsuche nur auf dem Teil des Graphen ausgeführt wird, der den Filter erfüllt. Das ist effizienter als erst alles zu suchen und dann 90% der Ergebnisse wegzuwerfen (Post-Filtering).

In [ ]:
# Erstelle den notwendigen Index für das Filtern
client.create_payload_index(
    collection_name=collection_name,
    field_name="topic",
    field_schema=PayloadSchemaType.KEYWORD,
)
client.create_payload_index(
    collection_name=collection_name,
    field_name="type",
    field_schema=PayloadSchemaType.KEYWORD,
)

In [ ]:
query_text = "Was für eine Tierart sind Blauwale"
query_vector = model.encode(query_text).tolist()

# 1. Definieren des Filters (damit wir ihn zweimal nutzen können)
my_filter = Filter(
    must=[
        # FieldCondition(key="topic", match=MatchValue(value="Sport")),
        FieldCondition(key="type", match=MatchValue(value="text"))
    ]
)
count_result = client.count(
    collection_name=collection_name,
    count_filter=my_filter
)

# 3. Die eigentliche Suche mit demselben Filter
res_pre = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=my_filter,
    with_payload=True,
    limit=3
).points

print(f"🔍 PRE-FILTERING ANALYSE")
print(f"Suchbegriff: '{query_text}'")
print(f"Gesamtanzahl Punkte in Collection: {client.get_collection(collection_name).points_count}")
print(f"Vektoren im 'Recall' nach Filter (nur Text): {count_result.count}")
print("-" * 50)

if count_result.count > 0:
    for hit in res_pre:
        print(f" - [{hit.payload.get('type')}] {hit.payload.get('topic')}: {hit.payload.get('content')[:50]}... (Score: {hit.score:.4f})")
else:
    print("Keine Dokumente gefunden, die den Filterkriterien entsprechen.")

In [ ]:
# 1. Globale Suche (keine Filter in Qdrant)
raw_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=10
).points

# 2. Filtern in Python auf Topic UND Type
res_post = [
    hit for hit in raw_results
    if hit.payload.get("topic") == "Sport" and hit.payload.get("type") == "text"
]

print(f"POST-FILTERING (Gefiltert in Python):")
for hit in res_post[:3]:
    print(f" - [{hit.payload.get('type')}] {hit.payload.get('topic')}: {hit.payload.get('content')[:50]}...")

print(f"\nInfo: Von {len(raw_results)} Treffern blieben {len(res_post)} Dokumente übrig.")

## 🎓 Fazit für die Präsentation

1.  **Vektoren sind das „Gedächtnis“:** Sie erlauben uns, nach Bedeutung statt nach Schlagworten zu suchen.
2.  **Effizienz durch ANN:** Für skalierbare Systeme ist HNSW unverzichtbar.
3.  **Hybrid-Suche:** Die Kombination aus Vektoren und Metadaten-Filtern macht Vektordatenbanken erst praxistauglich.